## D-Wave Minor Embedding for $^{20}\text{O}$ — NxD (1+1 particle) Encoding

### Imports

In [1]:
import numpy as np
import json
import matplotlib.pyplot as plt
from collections import defaultdict

from src.interaction_utils import (
    EffectiveInteractionOptimizer,
    EffectiveInteractionOptimizer2ndVersion,
)
from ManyBodyQutip.qutip_class import SpinOperator, SpinHamiltonian
import qutip as qt
from src.utils import computational_basis
from NSMFermions.hamiltonian_utils import FermiHubbardHamiltonian
from NSMFermions.nuclear_physics_utils import (
    get_twobody_nuclearshell_model,
    SingleParticleState,
)
from NSMFermions.utils_quasiparticle_approximation import (
    QuasiParticlesConverter,
    HardcoreBosonsBasis,
    QuasiParticlesConverterOnlynnpp,
)
from src.utils import (
    generate_particleconservation_basis,
    array_to_qutip,
    build_total_hamiltonian,
    build_effective_hamiltonian,
    compute_particle_number,
)

import dwave_networkx as dnx
import networkx as nx
import minorminer

/home/ecosta/miniconda3/envs/nsm_gadget_env/lib/python3.11/site-packages/tqdm_joblib/__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


### Load couplings and reproduce gadget quantities

We load the same one-body and two-body matrix elements as in `GadgetO20.ipynb` and recompute `d_opt`, `c_matrix`, and `links` (the self-energy shifts $\theta_A$) so this notebook is self-contained.

In [2]:
# ── One-body couplings (sd-shell, 6 orbitals) ──────────────────────────────
data_onebody = np.load("data/matrix_elements_h_eff_2body/one_body_nn_sd.npz")
keys = data_onebody["keys"]
values = data_onebody["values"]
n_sector = 6  # number of orbitals = qubits per particle sector

g_onebody = {}
diagonal_elements = np.zeros(n_sector)
g_matrix = np.zeros((n_sector, n_sector))

for a, key in enumerate(keys):
    i, j = key
    g_onebody[(i, j)] = values[a]
    if i != j:
        g_matrix[i, j] = values[a]
    else:
        diagonal_elements[i] = values[a]

# ── Two-body couplings ──────────────────────────────────────────────────────
data_twobody = np.load("data/matrix_elements_h_eff_2body/twobody_nn_sd.npz")
keys = data_twobody["keys"]
values = data_twobody["values"]
g_twobody = {}
for a, key in enumerate(keys):
    i, j, k, l = key
    g_twobody[(i, j)] = values[a]

print("diagonal_elements:", diagonal_elements)
print("g_matrix (off-diagonal):\n", g_matrix)
print("g_twobody:", g_twobody)

diagonal_elements: [-9.3151 -8.8615 -9.145  -8.1071  3.2251  3.2251]
g_matrix (off-diagonal):
 [[ 0.          0.6898     -0.4063     -0.90078189 -1.93144568  0.60173496]
 [ 0.6898      0.          0.8599      0.90078189  1.13361925 -1.39956139]
 [-0.4063      0.8599      0.         -0.90078189 -0.73470604  1.79847461]
 [-0.90078189  0.90078189 -0.90078189  0.         -0.71771338  0.71771338]
 [-1.93144568  1.13361925 -0.73470604 -0.71771338  0.          0.9009    ]
 [ 0.60173496 -1.39956139  1.79847461  0.71771338  0.9009      0.        ]]
g_twobody: {(np.int64(0), np.int64(1)): np.float64(-1.9616000000000007), (np.int64(0), np.int64(2)): np.float64(-1.394600000000004), (np.int64(0), np.int64(3)): np.float64(0.02873333333332795), (np.int64(0), np.int64(4)): np.float64(-2.203928571428574), (np.int64(0), np.int64(5)): np.float64(0.058295238095237956), (np.int64(1), np.int64(2)): np.float64(-2.3017999999999965), (np.int64(1), np.int64(3)): np.float64(0.0287333333333315), (np.int64(1), np.

### Reference: quasiparticle Hamiltonian in the two-body (hardcore boson) sector

This is the target $H_Q$ that the embedding must reproduce in the low-energy sector.

In [3]:
# Build the exact two-body quasiparticle Hamiltonian (N=2 sector)
nparticles = 2
particle_conserved_basis = generate_particleconservation_basis(
    size_a=n_sector, size_b=0, nparticles_a=nparticles, nparticles_b=0
)
HBB = HardcoreBosonsBasis(basis=particle_conserved_basis)

H_qp = 0.0
for key, value in g_onebody.items():
    idx_a, idx_b = key
    H_qp += value * HBB.adag_a_matrix(idx_a, idx_b)
for key, value in g_twobody.items():
    idx_a, idx_b = key
    H_qp += value * HBB.adag_adag_a_a_matrix(idx_a, idx_b, idx_a, idx_b)

eigenvalues_exact, eigenstates_particle_conserved = np.linalg.eigh(H_qp.todense())
print("Exact two-body spectrum (first 6):", eigenvalues_exact[:6])
print("Ground state energy:", eigenvalues_exact[0])

Exact two-body spectrum (first 6): [-23.42493918 -20.52242071 -19.67274996 -18.26654306 -16.81194491
 -15.9140891 ]
Ground state energy: -23.424939176438905


/home/ecosta/miniconda3/envs/nsm_gadget_env/lib/python3.11/site-packages/numba/typed/typeddict.py:39: NumbaTypeSafetyWarning: unsafe cast from int64 to uint64. Precision may be lost.
  return d[key]


### Optimize drive parameters $d_A$ and correction matrix $c_{AB}$

Stage 1: rank-1 fit of $d_A$ via `EffectiveInteractionOptimizer`.  
Stage 2: per-pair correction $c_{AB}$ via `EffectiveInteractionOptimizer2ndVersion`.

The convention is $\alpha_{AB} = \frac{1}{2}\left(1 + \frac{1}{1+c_{AB}}\right)$, so $c_{AB}=0$ is the no-correction baseline.

In [4]:
# Stage 1: rank-1 fit for d_opt
opt1 = EffectiveInteractionOptimizer(
    nqubit=n_sector, n_restarts=100, scale=2.0, ftol=1e-15, gtol=1e-10
)
d_opt, _ = opt1.optimize_rank1(g_matrix)
print("Optimal drive parameters d_opt:", d_opt)

# Stage 2: correction matrix c_AB
opt2 = EffectiveInteractionOptimizer2ndVersion(n_sector)
d_opt, _ = opt2.optimize_rank1(g_matrix)  # refit d_opt consistently
alpha_matrix, c_matrix, report, g_corrected = opt2.get_alpha_and_c(g_matrix, d_opt)
opt2.print_report(g_matrix, d_opt, alpha_matrix, c_matrix, report)

# Clip extreme corrections (same as GadgetO20.ipynb)
c_matrix[np.abs(c_matrix) > 2] = np.abs(c_matrix[np.abs(c_matrix) > 2])

print("\nc_matrix:\n", c_matrix)
print("alpha_matrix:\n", alpha_matrix)

Optimal drive parameters d_opt: [-0.90513022  1.02535362 -0.96153052 -0.79417758 -1.10508836  1.13502171]
d (fixed ansatz): [-0.9051  1.0254 -0.9615 -0.7942 -1.1051  1.135 ]
Rank-1-only loss: 2.242931e+00

  pair     target      rank1    alpha_AB        c_AB   pole?                          branch
----------------------------------------------------------------------------------------------------
  (0,1)     0.6898     0.9281      0.7433      1.0554          alpha<1 (shrink/flip vs rank-1)
  (0,2)    -0.4063    -0.8703      0.4668    -16.0807          alpha<1 (shrink/flip vs rank-1)
  (0,3)    -0.9008    -0.7188      1.2531     -0.3361               alpha>1 (boost vs rank-1)
  (0,4)    -1.9314    -1.0002      1.9310     -0.6506               alpha>1 (boost vs rank-1)
  (0,5)     0.6017     1.0273      0.5857      4.8329          alpha<1 (shrink/flip vs rank-1)
  (1,2)     0.8599     0.9859      0.8722      0.3434          alpha<1 (shrink/flip vs rank-1)
  (1,3)     0.9008     0.8143   

### Compute self-energy shifts $\theta_A$ (variable `links`)

The self-energy is extracted from the second-order BW correction in the single-particle sector. It is then used to shift the two-body interaction so all inter-sector $N_A N_B$ couplings are positive (required for Rydberg platforms; also convenient for D-Wave).

In [5]:
gamma = 100
ntot = 1
n_qubits_single = n_sector  # 6-qubit space for self-energy extraction
basis_single = computational_basis(n_qubits_single)

links = np.zeros(n_sector)

coupling_dict = {}
for i in range(n_sector):
    for j in range(i + 1, n_sector):
        if j != i + 1:
            coupling_dict[(i, j)] = 0.0

total_ham_single, _, _ = build_total_hamiltonian(
    n_qubits=n_qubits_single,
    d_opt=d_opt,
    gamma=gamma,
    links=links,
    coupling_dict=coupling_dict,
    ntot=ntot,
)

_, _, hamiltonian_delta, _, _ = build_effective_hamiltonian(
    total_hamiltonian=total_ham_single,
    basis=basis_single,
    gamma=gamma,
    low_energy_k=1,
    high_energy_k=[0, 2],
)

for i in range(n_sector):
    links[i] = float(
        np.real(hamiltonian_delta[-i - 1 + n_sector, -i - 1 + n_sector] * gamma)
    )

print("Self-energy shifts (links / theta_A):", links)

# Shifted two-body couplings: g_tilde_AB = g_AB + theta_A + theta_B
print("\nShifted two-body couplings (g_tilde_AB = g_AB + theta_A + theta_B):")
for (i, j), gij in g_twobody.items():
    print(
        f"  ({i},{j}): g={gij:.4f}, theta_i+theta_j={links[i]+links[j]:.4f}, "
        f"g_tilde={gij+links[i]+links[j]:.4f}"
    )

H_AA block:
<Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 0 stored elements and shape (6, 6)>

H_RR diagonal: [100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j
 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j 100.+0.j]

Second-order correction (in units of 1/gamma):
<Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 36 stored elements and shape (6, 6)>
  Coords	Values
  (0, 1)	(-1.2542992757838054+0j)
  (0, 2)	(-0.9014087954219213+0j)
  (0, 3)	(-1.0913580136979149+0j)
  (0, 4)	(1.163798623371701+0j)
  (0, 5)	(-1.027342444940011+0j)
  (0, 0)	(2.9676821450277258+0j)
  (1, 0)	(-1.2542992757838054+0j)
  (1, 2)	(0.8776363980404797+0j)
  (1, 3)	(1.0625761818378163+0j)
  (1, 4)	(-1.1331063520212565+0j)
  (1, 5)	(1.0002488632354922+0j)
  (1, 1)	(2.9676821450277253+0j)
  (2, 0)	(-0.9014087954219213+0j)
  (2, 1)	(0.8776363980404797+0j)
  (2, 3)	(0.7636259819379361+0j)
  (2, 4)	(-0.8143128610372079+0j)
  (2, 5)	(0.718834125426

### Minor embedding on Pegasus

We build the **logical connectivity graph** for the full 12-qubit NxD encoding:
- **Intra-sector edges**: full $K_6$ within sector 1 (qubits 0–5) and sector 2 (qubits 6–11) — needed for both the one-hot constraint ZZ terms and the $c_{AB}$ corrections.
- **Hardcore boson edges**: diagonal pairs $(i, i+6)$ for $i=0,\ldots,5$.
- **Two-body edges**: cross-sector pairs $(i, j+6)$ and $(j, i+6)$ for $i < j$ — these carry the $\tilde{g}^{(2)}_{AB}$ interactions.

In [6]:
n_logical = 12  # total logical qubits

G_logical = nx.Graph()
G_logical.add_nodes_from(range(n_logical))

# Full K6 within sector 1 (qubits 0..5)
for i in range(n_sector):
    for j in range(i + 1, n_sector):
        G_logical.add_edge(i, j)

# Full K6 within sector 2 (qubits 6..11)
for i in range(n_sector, n_logical):
    for j in range(i + 1, n_logical):
        G_logical.add_edge(i, j)

# Hardcore boson: diagonal pairs (i, i+n_sector)
for i in range(n_sector):
    G_logical.add_edge(i, i + n_sector)

# Two-body interaction: cross-sector pairs (i, j+n_sector) and (j, i+n_sector) for i < j
for i in range(n_sector):
    for j in range(i + 1, n_sector):
        G_logical.add_edge(i, j + n_sector)
        G_logical.add_edge(j, i + n_sector)

print(
    f"Logical graph: {G_logical.number_of_nodes()} nodes, "
    f"{G_logical.number_of_edges()} edges"
)

Logical graph: 12 nodes, 66 edges


In [7]:
# Find minor embedding on Pegasus P16
P16 = dnx.pegasus_graph(16)
embedding = minorminer.find_embedding(G_logical, P16, random_seed=42)

if not embedding:
    raise RuntimeError("minorminer failed — try again with a different random_seed")

chain_lengths = {l: len(c) for l, c in embedding.items()}
print("Chain lengths:", chain_lengths)
print(f"Max chain length: {max(chain_lengths.values())}")
print(f"Total physical qubits: {sum(chain_lengths.values())}")

Chain lengths: {0: 2, 1: 2, 2: 2, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2, 11: 2}
Max chain length: 2
Total physical qubits: 24


In [8]:
# Map physical qubit labels → compact local indices 0..n_physical-1
used_nodes = sorted({p for chain in embedding.values() for p in chain})
phys_to_idx = {phys: idx for idx, phys in enumerate(used_nodes)}
n_physical = len(used_nodes)

print(f"n_physical = {n_physical}")

# Chains in local indices
chains = {
    logical: [phys_to_idx[p] for p in chain] for logical, chain in embedding.items()
}

# Representatives: chains[A][0] is the canonical qubit for logical A
representatives = {logical: chain[0] for logical, chain in chains.items()}

# Reverse map: physical → logical
phys_to_logical = {}
for logical, chain in chains.items():
    for p in chain:
        phys_to_logical[p] = logical

# All edges among used nodes in local indices
P16_sub = P16.subgraph(used_nodes)
local_edges = [(phys_to_idx[u], phys_to_idx[v]) for u, v in P16_sub.edges()]

# Split into intra-chain (ferromagnetic) and inter-chain (logical coupling)
intra_set = set()
for logical, chain in chains.items():
    for k in range(len(chain) - 1):
        intra_set.add(frozenset((chain[k], chain[k + 1])))

intra_edges = [(a, b) for fset in intra_set for a, b in [tuple(fset)]]
inter_edges = [(u, v) for u, v in local_edges if frozenset((u, v)) not in intra_set]

print("\n=== Chains (local indices) ===")
for logical, chain in chains.items():
    sector = "S1" if logical < n_sector else "S2"
    print(f"  q{logical:2d} [{sector}] → {chain}  (len {len(chain)})")

print(f"\nIntra-chain (ferro) edges: {len(intra_edges)}")
print(f"Inter-chain (logical) edges: {len(inter_edges)}")
print(f"Representatives: {representatives}")

n_physical = 24

=== Chains (local indices) ===
  q 0 [S1] → [10, 11]  (len 2)
  q 1 [S1] → [5, 19]  (len 2)
  q 2 [S1] → [17, 6]  (len 2)
  q 3 [S1] → [16, 7]  (len 2)
  q 4 [S1] → [22, 2]  (len 2)
  q 5 [S1] → [8, 9]  (len 2)
  q 6 [S2] → [0, 21]  (len 2)
  q 7 [S2] → [23, 3]  (len 2)
  q 8 [S2] → [15, 14]  (len 2)
  q 9 [S2] → [13, 12]  (len 2)
  q10 [S2] → [1, 20]  (len 2)
  q11 [S2] → [4, 18]  (len 2)

Intra-chain (ferro) edges: 12
Inter-chain (logical) edges: 88
Representatives: {0: 10, 1: 5, 2: 17, 3: 16, 4: 22, 5: 8, 6: 0, 7: 23, 8: 15, 9: 13, 10: 1, 11: 4}


In [9]:
# Build logical-pair → physical-coupler map (one coupler per logical pair, dedup)
logical_pair_to_coupler = {}
added_pairs = set()

for u, v in inter_edges:
    lu = phys_to_logical[u]
    lv = phys_to_logical[v]
    if lu == lv:
        continue
    pair = frozenset((lu, lv))
    if pair in added_pairs:
        continue
    added_pairs.add(pair)
    logical_pair_to_coupler[pair] = (u, v)

print(f"Logical pairs with a coupler: {len(logical_pair_to_coupler)}")

# Classify each logical pair by type
intra_s1, intra_s2, hardcore, twobody = [], [], [], []
for pair in logical_pair_to_coupler:
    lu, lv = tuple(pair)
    A, B = min(lu, lv), max(lu, lv)
    if A < n_sector and B < n_sector:
        intra_s1.append(pair)
    elif A >= n_sector and B >= n_sector:
        intra_s2.append(pair)
    elif B - A == n_sector:  # diagonal: (i, i+6)
        hardcore.append(pair)
    else:
        twobody.append(pair)

print(f"  Intra-sector 1 (one-hot K6): {len(intra_s1)}")
print(f"  Intra-sector 2 (one-hot K6): {len(intra_s2)}")
print(f"  Hardcore boson (diagonal):   {len(hardcore)}")
print(f"  Two-body cross-sector:       {len(twobody)}")

Logical pairs with a coupler: 66
  Intra-sector 1 (one-hot K6): 15
  Intra-sector 2 (one-hot K6): 15
  Hardcore boson (diagonal):   6
  Two-body cross-sector:       30


### Build the physical Hamiltonian on the embedded qubits

The full physical Hamiltonian is:
$$H = H_C^{(1)} + H_C^{(2)} + H_{\rm HC} + H_{2b} + H_{\rm diag} + H_{\rm ferro} + H_X$$

where each piece is placed on the physical qubits according to the embedding:
- **$H_C^{(1,2)}$**: one-hot constraint ZZ for each sector — coupling $(2 + c_{AB})\gamma$ on inter-chain coupler for each intra-sector logical pair.
- **$H_{\rm HC}$**: hardcore boson — coupling $+\gamma$ on the coupler for each diagonal pair $(i, i+6)$.
- **$H_{2b}$**: two-body interaction — coupling $\tilde{g}^{(2)}_{AB}/\gamma$ on each cross-sector coupler.
- **$H_{\rm diag}$**: diagonal energy $\epsilon_A/\gamma$ distributed over all qubits in chain $A$.
- **$H_{\rm ferro}$**: ferromagnetic coupling $-J_F$ on all intra-chain edges.
- **$H_X$**: transverse drive $d_A/\sqrt{2}$ on the representative of each logical qubit.

In [ ]:
gamma = 100
J_F = 10 * gamma
N_tot = 1  # per-sector particle number
n_qubits = n_logical  # rename for compatibility

identity = qt.tensor([qt.qeye(2)] * n_physical)

# ── (a) Ferromagnetic chains ────────────────────────────────────────────────
H_ferro = 0.0
for a, b in intra_edges:
    H_ferro += SpinOperator(
        [("z", a, "z", b)], coupling=[-J_F], size=n_physical
    ).qutip_op
    H_ferro += (-J_F) * identity  # constant offset per bond

# ── (b) ZZ couplings (all logical pair types) ───────────────────────────────
H_zz = 0.0
for pair, (u, v) in logical_pair_to_coupler.items():
    lu, lv = tuple(pair)
    A, B = min(lu, lv), max(lu, lv)

    if A < n_sector and B < n_sector:
        # Intra-sector 1 one-hot: (2 + c_AB) * gamma
        J = (2.0 + c_matrix[A, B]) * gamma

    elif A >= n_sector and B >= n_sector:
        # Intra-sector 2 one-hot: same c_matrix, shifted indices
        ia, ib = A - n_sector, B - n_sector
        J = (2.0 + c_matrix[ia, ib]) * gamma

    elif B - A == n_sector:
        # Hardcore boson: diagonal pair (i, i+n_sector) → +gamma
        J = gamma

    else:
        # Two-body cross-sector: (A, B) where one < n_sector, other >= n_sector
        i = A if A < n_sector else (B - n_sector)
        j = (B - n_sector) if A < n_sector else A
        # ensure canonical ordering for g_twobody lookup
        key = (min(i, j), max(i, j))
        g_tilde = g_twobody[key] + links[i] + links[j]
        J = g_tilde / gamma

    H_zz += SpinOperator([("qz", u, "qz", v)], coupling=[J], size=n_physical).qutip_op

# ── (c) Linear terms on representatives (constraint + diagonal energy) ──────
h_constraint = gamma * (1 - 2 * N_tot)  # = -gamma for N_tot=1

H_linear = 0.0
for logical in range(n_logical):
    sector_idx = logical % n_sector
    rep = chains[logical][0]  # representative physical qubit
    k = len(chains[logical])  # chain length

    # One-hot constraint linear term on representative only
    H_linear += SpinOperator(
        [("qz", rep)], coupling=[h_constraint], size=n_physical
    ).qutip_op

    # Diagonal energy epsilon_A / (gamma * k) distributed over all chain qubits
    eps = diagonal_elements[sector_idx] / (gamma * k)
    for phys in chains[logical]:
        H_linear += SpinOperator(
            [("qz", phys)], coupling=[eps], size=n_physical
        ).qutip_op

# ── (d) Identity offset (2 sectors × gamma × N_tot²) ───────────────────────
H_longitudinal = H_zz + H_linear + H_ferro + 2 * gamma * (N_tot**2) * identity

# ── (e) Transverse field (same d_opt duplicated for both sectors) ────────────
H_transverse = 0.0
for logical in range(n_logical):
    sector_idx = logical % n_sector
    rep = chains[logical][0]
    H_transverse += SpinOperator(
        [("x", rep)], coupling=[d_opt[sector_idx] / np.sqrt(2)], size=n_physical
    ).qutip_op

H_total_phys = H_longitudinal + H_transverse

print(f"gamma={gamma}, J_F={J_F}, N_tot={N_tot}")
print(f"n_physical={n_physical}, n_logical={n_logical}")

### Diagnostics: check diagonal structure

The longitudinal Hamiltonian (without transverse field) should have its lowest-energy states in the **valid two-particle subspace**: one particle in sector 1 and one in sector 2, with distinct orbital indices (hardcore boson condition).

In [ ]:
print("=== Longitudinal Hamiltonian: lowest 20 diagonal entries ===")
diag_long = np.real(H_longitudinal.full().diagonal())
idx_sorted = np.argsort(diag_long)

for k in idx_sorted[:20]:
    bitstring = format(k, f"0{n_physical}b")
    # decode to logical occupations via chains
    logical_occupations = []
    for logical, chain in chains.items():
        # a logical qubit is 'occupied' if ALL its chain qubits are 1 (ferro-aligned)
        if all((k >> (n_physical - 1 - p)) & 1 for p in chain):
            logical_occupations.append(logical)
    n_s1 = sum(1 for q in logical_occupations if q < n_sector)
    n_s2 = sum(1 for q in logical_occupations if q >= n_sector)
    valid = (
        n_s1 == 1
        and n_s2 == 1
        and len(logical_occupations) == 2
        and logical_occupations[0] != logical_occupations[1] - n_sector
    )
    tag = " ✓ valid" if valid else ""
    print(
        f"  E={diag_long[k]:10.4f} | N_s1={n_s1} N_s2={n_s2} | "
        f"logical={logical_occupations}{tag}"
    )

In [ ]:
# Full spectrum of H_total_phys (dense diagonalization)
evals_phys, evecs_phys = np.linalg.eigh(H_total_phys.full())
print("Physical Hamiltonian spectrum (first 10):", evals_phys[:10])
print(
    "Ground state degeneracy:", np.sum(np.isclose(evals_phys, evals_phys[0], atol=1e-8))
)

### Decode the embedded ground state

We project the physical ground state onto the valid **NxD logical subspace** (one particle per sector, no same-site occupation). Each logical two-body basis state $|i,j\rangle$ corresponds to chain $i$ in sector 1 AND chain $j$ in sector 2 all-ones (ferro-aligned), with $i \neq j$.

In [ ]:
def qutip_chain_idx(chains_list, n_phys):
    """
    Return the computational basis index (in QuTiP MSB convention)
    for the state where all qubits in chains_list are 1 and rest are 0.
    chains_list: list of physical qubit indices (local 0..n_phys-1)
    """
    idx = 0
    for p in chains_list:
        idx += 1 << (n_phys - 1 - p)
    return idx


def decode_to_logical(gs_phys, chains, n_sector, n_physical):
    """
    Project the physical ground state onto the valid NxD logical subspace.
    Returns a dict: (i, j) -> amplitude, where i in 0..n_sector-1 (sector 1 orbital)
    and j in 0..n_sector-1 (sector 2 orbital), i != j.
    Leakage is captured by the norm of the returned dict being < 1.
    """
    psi = gs_phys.full().flatten()
    amplitudes = {}
    for i in range(n_sector):  # sector 1 orbital
        for j in range(n_sector):  # sector 2 orbital
            if i == j:
                continue  # hardcore boson: no same-site
            # physical qubits: chain of sector-1 orbital i + chain of sector-2 orbital j
            occupied = chains[i] + chains[j + n_sector]
            idx = qutip_chain_idx(occupied, n_physical)
            amplitudes[(i, j)] = psi[idx]
    return amplitudes


# Ground state of the physical Hamiltonian
gs_phys_idx = 0  # lowest eigenvalue
gs_phys = qt.Qobj(evecs_phys[:, gs_phys_idx], dims=[[2] * n_physical, [1] * n_physical])

emb_amps_dict = decode_to_logical(gs_phys, chains, n_sector, n_physical)

norm_emb = sum(np.abs(a) ** 2 for a in emb_amps_dict.values())
print(f"Norm of projected embedded state: {norm_emb:.6f}")
print("(< 1 means weight leaked to invalid / multi-particle states)")

### Fidelity: embedded vs exact two-body ground state

We compare the embedded amplitudes (projected onto the NxD logical subspace) against the exact amplitudes from `H_qp` in the hardcore-boson basis. The fidelity uses the **unnormalized** embedded state so that leakage appears directly as fidelity loss.

In [ ]:
# Build exact amplitudes in the same (i,j) ordering
exact_gs = eigenstates_particle_conserved[:, 0]  # ground state column

exact_amps_dict = {}
for r, amp in enumerate(exact_gs):
    idxs = tuple(np.nonzero(HBB.basis[r])[0])  # e.g. (i, j)
    if len(idxs) == 2:
        i, j = idxs
        # NxD convention: (i, j) means orbital i in sector 1, orbital j in sector 2
        # HBB basis is symmetric, so we store both orderings
        exact_amps_dict[(i, j)] = amp
        exact_amps_dict[(j, i)] = amp  # symmetry (will be 0 in practice for HCB)

# Compute overlap (unnormalized fidelity)
overlap = 0.0 + 0.0j
for key, amp_emb in emb_amps_dict.items():
    amp_exact = exact_amps_dict.get(key, 0.0)
    overlap += np.conj(amp_exact) * amp_emb

fidelity_val = np.abs(overlap) ** 2

print(f"Fidelity (raw, penalises leakage): {fidelity_val:.6f}")
print(f"Norm of embedded state:            {norm_emb:.6f}")
print(
    f"Norm of exact state:               {sum(np.abs(a)**2 for a in exact_amps_dict.values()):.6f}"
)

### Amplitude comparison table

In [ ]:
# Phase-align embedded to exact (using the dominant component)
dominant_key = max(exact_amps_dict, key=lambda k: np.abs(exact_amps_dict[k]))

phase_exact = np.angle(exact_amps_dict[dominant_key])
phase_emb = np.angle(emb_amps_dict.get(dominant_key, 1.0 + 0j))

print(
    f"{'Pair (i,j)':>12}  {'Re(exact)':>10}  {'Re(emb)':>10}  "
    f"{'|exact|²':>9}  {'|emb|²':>9}"
)
print("-" * 60)

for i in range(n_sector):
    for j in range(n_sector):
        if i == j:
            continue
        key = (i, j)
        a_ex = exact_amps_dict.get(key, 0.0) * np.exp(-1j * phase_exact)
        a_emb = emb_amps_dict.get(key, 0.0) * np.exp(-1j * phase_emb)
        print(
            f"  ({i},{j})       "
            f"{a_ex.real:>10.5f}  {a_emb.real:>10.5f}  "
            f"{np.abs(a_ex)**2:>9.5f}  {np.abs(a_emb)**2:>9.5f}"
        )

### Visualisation: embedded vs exact amplitudes

In [ ]:
pairs = [(i, j) for i in range(n_sector) for j in range(n_sector) if i != j]
labels = [f"({i},{j})" for i, j in pairs]

phase_exact = np.angle(exact_amps_dict.get(dominant_key, 1.0 + 0j))
phase_emb = np.angle(emb_amps_dict.get(dominant_key, 1.0 + 0j))

exact_vals = np.array(
    [(exact_amps_dict.get(p, 0.0) * np.exp(-1j * phase_exact)).real for p in pairs]
)
emb_vals = np.array(
    [(emb_amps_dict.get(p, 0.0) * np.exp(-1j * phase_emb)).real for p in pairs]
)

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(pairs))
w = 0.35
CLR_E = "#2775b6"
CLR_M = "#d85a30"

ax.bar(x - w / 2, exact_vals, w, color=CLR_E, alpha=0.85, label="Exact")
ax.bar(x + w / 2, emb_vals, w, color=CLR_M, alpha=0.85, label="Embedded")
ax.axhline(0, color="k", linewidth=0.7)
ax.set_ylabel(r"$\mathrm{Re}\,\langle i,j|\psi\rangle$", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7, rotation=45, ha="right", fontfamily="monospace")
ax.legend()
ax.set_title(
    f"Real part of amplitudes — NxD two-body sector\n"
    f"$\\gamma$={gamma},  $J_F$={J_F},  Fidelity={fidelity_val:.5f}",
    fontsize=12,
)
plt.tight_layout()
plt.savefig("wavefunction_real_O20.pdf", bbox_inches="tight")
plt.show()

print(f"\nFidelity = {fidelity_val:.6f}")
print(f"Norm of embedded state = {norm_emb:.6f}")

### Visualise the embedding on Pegasus

In [ ]:
# Color map: sector 1 → blue palette, sector 2 → red palette
import matplotlib.cm as cm

blues = cm.Blues(np.linspace(0.4, 0.9, n_sector))
reds = cm.Reds(np.linspace(0.4, 0.9, n_sector))

# Map each physical (local) qubit to a color
node_colors = {}
for logical, chain in chains.items():
    if logical < n_sector:
        color = blues[logical % n_sector]
    else:
        color = reds[(logical - n_sector) % n_sector]
    for p in chain:
        node_colors[p] = color

# Rebuild subgraph in physical labels for drawing
subgraph = P16.subgraph(used_nodes).copy()
phys_colors = [node_colors[phys_to_idx[n]] for n in subgraph.nodes()]

used_phys_set = set(used_nodes)
used_edges_phys = {
    frozenset((u, v))
    for u, v in P16_sub.edges()
    if phys_to_idx[u] in phys_to_logical
    and phys_to_idx[v] in phys_to_logical
    and phys_to_logical[phys_to_idx[u]] != phys_to_logical[phys_to_idx[v]]
}

pos = dnx.pegasus_layout(subgraph)
fig, ax = plt.subplots(figsize=(8, 8))
nx.draw(
    subgraph,
    pos,
    node_color=phys_colors,
    edge_color=[
        (
            "black"
            if frozenset((u, v)) in used_edges_phys
            or frozenset((v, u)) in used_edges_phys
            else "lightgray"
        )
        for u, v in subgraph.edges()
    ],
    node_size=250,
    with_labels=False,
    ax=ax,
)

# Legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=blues[i], label=f"q{i} (S1)") for i in range(n_sector)
] + [Patch(facecolor=reds[i], label=f"q{i+n_sector} (S2)") for i in range(n_sector)]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8, ncol=2)
ax.set_title(
    f"$^{{20}}$O embedding on Pegasus P16\n"
    f"{n_physical} physical qubits, max chain={max(chain_lengths.values())}"
)
plt.tight_layout()
plt.savefig("embedding_O20.png", dpi=150)
plt.show()

### Fidelity vs $\gamma$ scan

Sweep $\gamma$ to find the crossover where the gadget approximation becomes accurate.

In [ ]:
gammas = np.linspace(5, 300, 30)
fidelities = []
norms_emb = []

for gam in gammas:
    J_F_scan = 10 * gam
    identity_scan = qt.tensor([qt.qeye(2)] * n_physical)

    # Ferro
    H_ferro_scan = 0.0
    for a, b in intra_edges:
        H_ferro_scan += SpinOperator(
            [("z", a, "z", b)], coupling=[-J_F_scan], size=n_physical
        ).qutip_op
        H_ferro_scan += (-J_F_scan) * identity_scan

    # ZZ
    H_zz_scan = 0.0
    for pair, (u, v) in logical_pair_to_coupler.items():
        lu, lv = tuple(pair)
        A, B = min(lu, lv), max(lu, lv)
        if A < n_sector and B < n_sector:
            J = (2.0 + c_matrix[A, B]) * gam
        elif A >= n_sector and B >= n_sector:
            ia, ib = A - n_sector, B - n_sector
            J = (2.0 + c_matrix[ia, ib]) * gam
        elif B - A == n_sector:
            J = gam
        else:
            i = A if A < n_sector else (B - n_sector)
            j = (B - n_sector) if A < n_sector else A
            key = (min(i, j), max(i, j))
            J = (g_twobody[key] + links[i] + links[j]) / gam
        H_zz_scan += SpinOperator(
            [("qz", u, "qz", v)], coupling=[J], size=n_physical
        ).qutip_op

    # Linear
    h_c = gam * (1 - 2 * N_tot)
    H_lin_scan = 0.0
    for logical in range(n_logical):
        sector_idx = logical % n_sector
        rep = chains[logical][0]
        k = len(chains[logical])
        H_lin_scan += SpinOperator(
            [("qz", rep)], coupling=[h_c], size=n_physical
        ).qutip_op
        eps = diagonal_elements[sector_idx] / (gam * k)
        for phys in chains[logical]:
            H_lin_scan += SpinOperator(
                [("qz", phys)], coupling=[eps], size=n_physical
            ).qutip_op

    H_long_scan = H_zz_scan + H_lin_scan + H_ferro_scan + 2 * gam * identity_scan

    # Transverse
    H_trans_scan = 0.0
    for logical in range(n_logical):
        sector_idx = logical % n_sector
        rep = chains[logical][0]
        H_trans_scan += SpinOperator(
            [("x", rep)], coupling=[d_opt[sector_idx] / np.sqrt(2)], size=n_physical
        ).qutip_op

    H_tot_scan = H_long_scan + H_trans_scan

    evals_s, evecs_s = np.linalg.eigh(H_tot_scan.full())
    gs_s = qt.Qobj(evecs_s[:, 0], dims=[[2] * n_physical, [1] * n_physical])

    amps_emb = decode_to_logical(gs_s, chains, n_sector, n_physical)
    norm_s = sum(np.abs(a) ** 2 for a in amps_emb.values())

    ov = sum(
        np.conj(exact_amps_dict.get(k, 0.0)) * amps_emb.get(k, 0.0) for k in amps_emb
    )
    fidelities.append(np.abs(ov) ** 2)
    norms_emb.append(norm_s)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(gammas, fidelities, "o-", color="#2775b6", linewidth=2, markersize=5)
ax1.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax1.set_xlabel(r"$\gamma / \omega$", fontsize=13)
ax1.set_ylabel(r"Fidelity $F(\gamma)$", fontsize=13)
ax1.set_title(r"$^{20}$O NxD embedding — fidelity vs $\gamma$", fontsize=12)
ax1.set_ylim(0, 1.05)
ax1.grid(alpha=0.3)

ax2.plot(gammas, norms_emb, "s-", color="#d85a30", linewidth=2, markersize=5)
ax2.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax2.set_xlabel(r"$\gamma / \omega$", fontsize=13)
ax2.set_ylabel(r"Norm of embedded state", fontsize=13)
ax2.set_title(r"Leakage out of NxD subspace", fontsize=12)
ax2.set_ylim(0, 1.05)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fidelity_vs_gamma_O20.pdf", bbox_inches="tight")
plt.show()